# Classificação de Pavimentos com Visão Computacional
 - João Pedro Barbosa Marins (jpbm)

## Identificação da Abordagem

Este trabalho aborda um problema de **classificação de imagens** com o objetivo de identificar o tipo de pavimento presente em imagens capturadas por uma câmera acoplada a um veículo. As classes consideradas são:

* `asphalt`
* `belgian_blocks`
* `offroad`

A solução proposta utiliza **redes neurais convolucionais (CNNs)** com **transfer learning**, aproveitando modelos pré-treinados para compensar o tamanho reduzido do dataset.

A escolha dessa abordagem se justifica por:

* CNNs serem o estado da arte para tarefas de visão computacional.
* Modelos pré-treinados permitirem melhor generalização em cenários com poucos dados.

**Bibliotecas utilizadas:**

* `TensorFlow / Keras`
* `NumPy`
* `Matplotlib`
* `Scikit-learn`

---

## Preparação e Execução

O conjunto de dados é composto por imagens reais, sujeitas a variações como iluminação, clima e perspectiva. O dataset já está dividido em:

* **Treino (75%)** — 900 imagens
* **Teste (25%)** — 300 imagens

### Distribuição dos Dados

#### Treino:

* **asphalt:** 655
* **belgian_blocks:** 94
* **offroad:** 151

#### Teste:

* **asphalt:** 218
* **belgian_blocks:** 32
* **offroad:** 50

---

## Experimento 1 (Baseline)

**Hipótese:**
Um modelo pré-treinado, mesmo com ajustes simples como o uso de pesos de classe, pode alcançar um desempenho inicial satisfatório.

**Configuração:**

* Modelo: CNN com **transfer learning (MobileNetV2)**
* Uso de **class weights** durante o treinamento para mitigar o desbalanceamento entre as classes
* Aplicação de **data augmentation** para aumentar a variabilidade dos dados:

  * `rotation_range=5`
  * `zoom_range=0.1`
  * `width_shift_range=0.05`
  * `height_shift_range=0.05`
  * `brightness_range=[0.8, 1.2]`
* Treinamento por **10 epochs**


---

## Carregamento e pré-processamento das imagens

In [ ]:
%pip install patool tensorflow pillow scipy numpy matplotlib scikit-learn

In [ ]:
import patoolib
patoolib.extract_archive("dataset_processed.zip",outdir="./content/")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.utils import class_weight
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import classification_report

np.random.seed(42)
tf.random.set_seed(42)

IMG_SIZE = 224
BATCH_SIZE = 16

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    rotation_range=5,
    zoom_range=0.1,
    width_shift_range=0.05,
    height_shift_range=0.05,
    brightness_range=[0.8, 1.2]
)

test_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input
)

train_data = train_datagen.flow_from_directory(
    'content/dataset_processed/train',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_data = test_datagen.flow_from_directory(
    'content/dataset_processed/test',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
y_train = train_data.classes

weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

classes = np.unique(y_train)
class_weights = dict(zip(classes, weights))

## Treinamento do modelo

In [ ]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Congelar base (primeira fase)
for layer in base_model.layers:
    layer.trainable = False

# Cabeça nova (classificador)
x = base_model.output
avg = keras.layers.GlobalAveragePooling2D()(base_model.output)
outputs = keras.layers.Dense(train_data.num_classes, activation="softmax")(avg)

model = models.Model(inputs=base_model.input, outputs=outputs)

# Compilar
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=["accuracy"]
)

# Treinar (fase 1)
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=10,
    class_weight=class_weights
)

In [ ]:
# Fine-tuning
for layer in base_model.layers[-20:]:  # Descongelar últimas 20 camadas
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # LR menor
    loss='categorical_crossentropy',
    metrics=["accuracy"]
)

history_finetune = model.fit(
    train_data,
    validation_data=test_data,
    epochs=10,
    class_weight=class_weights
)

## Avaliação no conjunto de teste

In [ ]:
def plot_confusion_matrix(model, data_generator, normalize=True, figsize=(8,6)):
    """
    Plota a matriz de confusão para um generator do Keras.

    Parâmetros:
    - model: modelo treinado
    - data_generator: generator (ex: flow_from_directory)
    - normalize: se True, mostra valores normalizados
    - figsize: tamanho da figura
    """

    # Reset para garantir ordem correta
    data_generator.reset()

    # Predições
    y_pred_probs = model.predict(data_generator, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)

    # Labels reais
    y_true = data_generator.classes

    # Nomes das classes
    class_names = list(data_generator.class_indices.keys())

    # Matriz de confusão
    cm = confusion_matrix(y_true, y_pred)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    # Plot
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

    fig, ax = plt.subplots(figsize=figsize)
    disp.plot(
        cmap='Blues',
        ax=ax,
        xticks_rotation=45,
        values_format='.2f' if normalize else 'd'
    )

    title = "Matriz de Confusão Normalizada" if normalize else "Matriz de Confusão"
    plt.title(title)
    plt.grid(False)
    plt.tight_layout()
    plt.show()

plot_confusion_matrix(model, test_data, normalize=True)

In [ ]:
def print_classification_report(model, data_generator):
    """
    Gera e imprime o classification report para um generator do Keras.
    """

    # Garantir ordem correta
    data_generator.reset()

    # Predições
    y_pred_probs = model.predict(data_generator, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)

    # Labels reais
    y_true = data_generator.classes

    # Nomes das classes
    class_names = list(data_generator.class_indices.keys())

    # Report
    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names
    )

    print(report)

    return report

print_classification_report(model, test_data)


**Resultado observado:**

* Boa performance geral (**accuracy: 0.84**)
* Forte desempenho na classe `asphalt` (F1: 0.91)
* Desempenho limitado na classe `belgian_blocks` (F1: 0.57, recall: 0.41)
* Alta capacidade de detecção da classe `offroad` (recall: 0.92), com menor precisão

**Interpretação:**

O modelo apresentou bom desempenho geral, especialmente na classe majoritária (`asphalt`). No entanto, ainda há dificuldade em identificar corretamente a classe `belgian_blocks`, evidenciada pelo baixo recall.

Apesar do uso de **class weights**, o desbalanceamento do dataset continua impactando o aprendizado. O modelo também apresenta tendência a classificar amostras como `offroad`, indicando possível confusão entre classes com características visuais semelhantes.


---

## Experimento 2 — Região de Interesse (ROI)

**Hipótese:**
Remover partes irrelevantes da imagem e manter apenas a região inferior (onde o pavimento está mais visível) pode melhorar a capacidade do modelo de extrair características discriminativas.

**Configuração:**

* Aplicação de **crop mantendo 40% da porção inferior da imagem (ROI)**
* Modelo: **MobileNetV2 (transfer learning)**
* Uso de **class weights**
* Uso de **data augmentation**
* Treinamento por **10 epochs**

**Controle experimental:**
Todas as demais configurações foram **mantidas iguais ao Experimento 1 (baseline)**, incluindo arquitetura da cabeça de classificação, hiperparâmetros e pipeline de treinamento. Dessa forma, o impacto observado nos resultados pode ser atribuído exclusivamente à aplicação do recorte da região de interesse.

---

In [ ]:
def preprocess_image(image):
    image = image[int(image.shape[0]*0.4):, :]
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    return image

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_image,
    rotation_range=5,
    zoom_range=0.1,
    width_shift_range=0.05,
    height_shift_range=0.05,
    brightness_range=[0.8, 1.2]
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_image
)

train_data = train_datagen.flow_from_directory(
    'content/dataset_processed/train',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_data = test_datagen.flow_from_directory(
    'content/dataset_processed/test',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
y_train = train_data.classes

weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(weights))

In [ ]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Congelar base (primeira fase)
for layer in base_model.layers:
    layer.trainable = False

# Cabeça nova (classificador)
x = base_model.output
avg = keras.layers.GlobalAveragePooling2D()(base_model.output)
outputs = keras.layers.Dense(train_data.num_classes, activation="softmax")(avg)

model = models.Model(inputs=base_model.input, outputs=outputs)

# Compilar
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=["accuracy"]
)

# Treinar (fase 1)
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=10,
    class_weight=class_weights
)

In [ ]:
# Fine-tuning
for layer in base_model.layers[-20:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # LR menor
    loss='categorical_crossentropy',
    metrics=["accuracy"]
)

history_finetune = model.fit(
    train_data,
    validation_data=test_data,
    epochs=10,
    class_weight=class_weights
)

In [ ]:
plot_confusion_matrix(model, test_data, normalize=True)

In [ ]:
print_classification_report(model, test_data)

**Resultado observado:**

* Desempenho geral semelhante ao baseline (**accuracy: 0.83**, F1 macro: 0.73)
* Melhora na classe `belgian_blocks`:

  * Recall: **0.41 → 0.47**
  * F1: **0.57 → 0.61**
* Pequenas variações nas demais classes

**Interpretação:**

O uso de ROI ajudou o modelo a focar em regiões mais relevantes da imagem, resultando em uma **melhora na detecção da classe minoritária (`belgian_blocks`)**.

No entanto, essa melhoria não se refletiu em ganho significativo no desempenho global. Isso sugere que, embora o recorte reduza ruído e direcione o aprendizado, ele também pode remover parte do contexto visual útil, limitando os ganhos quando utilizado com uma arquitetura menos complexa.


---

## Experimento 3 — ROI + ResNet50V2

**Hipótese:**
A combinação de uma entrada mais focada (ROI) com uma arquitetura mais profunda pode melhorar a extração de características complexas e aumentar o desempenho geral do modelo.

**Configuração:**

* Modelo: **ResNet50V2 (transfer learning)**
* Uso de **ROI (40% inferior da imagem)**
* Uso de **class weights**
* Uso de **data augmentation**
* Mesma cabeça de classificação dos experimentos anteriores
* Treinamento por **10 epochs**

**Controle experimental:**
Em relação ao Experimento 2, todas as configurações foram **mantidas constantes**, incluindo o uso de ROI, data augmentation, class weights e pipeline de treinamento. A única alteração realizada foi a substituição da arquitetura base (**MobileNetV2 → ResNet50V2**), permitindo isolar o impacto da capacidade do modelo nos resultados obtidos.

---

In [ ]:
from tensorflow.keras.applications import ResNet50

def preprocess_image(image):
    image = image[int(image.shape[0]*0.4):, :]
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.keras.applications.resnet.preprocess_input(image)
    return image

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_image,
    rotation_range=5,
    zoom_range=0.1,
    width_shift_range=0.05,
    height_shift_range=0.05,
    brightness_range=[0.8, 1.2]
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_image
)

train_data = train_datagen.flow_from_directory(
    'content/dataset_processed/train',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_data = test_datagen.flow_from_directory(
    'content/dataset_processed/test',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
y_train = train_data.classes

weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

classes = np.unique(y_train)
class_weights = dict(zip(classes, weights))

In [ ]:
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Congelar base (primeira fase)
for layer in base_model.layers:
    layer.trainable = False

# Cabeça nova (classificador)
x = base_model.output
avg = keras.layers.GlobalAveragePooling2D()(base_model.output)
outputs = keras.layers.Dense(train_data.num_classes, activation="softmax")(avg)

model = models.Model(inputs=base_model.input, outputs=outputs)

# Compilar
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=["accuracy"]
)

# Treinar (fase 1)
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=10,
    class_weight=class_weights
)

In [ ]:
# Fine-tuning
for layer in base_model.layers[-20:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # LR menor
    loss='categorical_crossentropy',
    metrics=["accuracy"]
)

history_finetune = model.fit(
    train_data,
    validation_data=test_data,
    epochs=10,
    class_weight=class_weights
)

In [ ]:
plot_confusion_matrix(model, test_data, normalize=True)

In [ ]:
print_classification_report(model, test_data)


**Resultado observado:**

* Melhor desempenho geral (**accuracy: 0.89**, F1 macro: 0.81)
* Excelente performance na classe `asphalt` (F1: 0.95)
* Melhora significativa na classe `belgian_blocks`:

  * Recall: **0.59**
  * F1: **0.69**
* Melhor equilíbrio geral entre precisão e recall nas classes

**Interpretação:**

A combinação de **ROI + arquitetura mais profunda** mostrou-se a abordagem mais eficaz. O modelo conseguiu extrair características mais discriminativas mesmo com a entrada reduzida, resultando em melhorias consistentes em todas as classes.

O ganho mais relevante ocorreu na classe `belgian_blocks`, indicando que a maior capacidade de representação da ResNet50V2 foi essencial para lidar com a baixa quantidade de dados e a complexidade visual dessa classe.

---

## Análise Crítica

### Experimento 1 — Baseline

O modelo baseline, baseado na MobileNetV2 com uso de *class weights*, apresentou um desempenho geral satisfatório. Observou-se alta performance na classe majoritária (`asphalt`), indicando que o modelo foi capaz de aprender bem padrões mais representativos no dataset.

Entretanto, o desempenho na classe `belgian_blocks` foi significativamente inferior, especialmente em termos de recall, evidenciando dificuldade em identificar corretamente essa classe. Isso reforça o impacto do **desbalanceamento do dataset**, que não foi completamente mitigado apenas com o uso de pesos de classe.

Além disso, o modelo apresentou tendência a classificar amostras como `offroad`, sugerindo confusão entre classes com características visuais semelhantes.

---

### Experimento 2 — Região de Interesse (ROI)

A introdução do recorte da **região de interesse (ROI)** teve como objetivo direcionar o modelo para áreas mais relevantes da imagem, reduzindo a influência de elementos irrelevantes.

Os resultados mostraram uma **melhora na detecção da classe `belgian_blocks`**, indicando que o foco na textura do pavimento contribui positivamente para o aprendizado. No entanto, essa melhoria não se refletiu em ganhos expressivos no desempenho global.

Uma possível explicação para esse comportamento está na **distorção introduzida pelo redimensionamento das imagens para 224x224 após o recorte**. Ao utilizar apenas 40% da imagem original e redimensioná-la, há uma amplificação da região selecionada, o que pode causar perda de detalhes finos e distorção das texturas, prejudicando a extração de características discriminativas.

Isso sugere que, embora o ROI reduza ruído, ele também pode remover ou degradar informações relevantes.

---

### Experimento 3 — ROI + ResNet50V2

A combinação do uso de ROI com uma arquitetura mais profunda (**ResNet50V2**) apresentou os melhores resultados entre todos os experimentos.

O modelo conseguiu alcançar um melhor equilíbrio entre as classes, com destaque para a melhora significativa na classe `belgian_blocks`. Isso indica que a maior capacidade de representação da ResNet50V2 foi essencial para capturar padrões mais complexos e compensar limitações introduzidas pelo pré-processamento.

Além disso, houve melhora geral nas métricas, demonstrando maior capacidade de generalização e redução de erros sistemáticos observados nos experimentos anteriores.

---

### Considerações Gerais

De forma geral, os experimentos demonstram que o desempenho do modelo depende fortemente da **interação entre qualidade dos dados e capacidade da arquitetura utilizada**.

* O baseline evidenciou o impacto do desbalanceamento
* O ROI mostrou-se útil, mas introduziu desafios relacionados à perda de informação
* A ResNet50V2 conseguiu explorar melhor os dados disponíveis, mesmo com limitações

---

### Limitações

* **Desbalanceamento do dataset**, especialmente na classe `belgian_blocks`
* **Baixa quantidade de dados**, limitando a generalização
* **Perda de informação devido ao pré-processamento** (crop + resize)
* **Semelhança visual entre classes**
* Possível **overfitting** em modelos mais complexos

---

### Trabalhos Futuros

* Ajustar a estratégia de ROI (ex: uso de padding ou cortar na mesma proporção do formato final)
* Coletar mais dados, principalmente para classes minoritárias
* Intensificar o uso de **data augmentation**
* Explorar técnicas mais avançadas de balanceamento
* Realizar **fine-tuning mais profundo**
* Avaliar possíveis alterações na estrutura do classificador

---

## Uso de Ferramentas

Durante o desenvolvimento da solução, foram utilizadas ferramentas de apoio para auxiliar na organização, implementação e análise do projeto.

Em particular, foi utilizado o **ChatGPT** como modelo de linguagem (LLM), auxiliando na **estruturação do notebook**, na **revisão da redação técnica** e na **interpretação dos resultados obtidos nos experimentos**.

Além disso, foram utilizadas bibliotecas amplamente adotadas em aprendizado de máquina, como **TensorFlow/Keras**, **Scikit-learn**, **NumPy** e **Matplotlib**, que contribuíram para a implementação dos modelos, avaliação de desempenho e visualização dos resultados.

Também foi utilizada como referência teórica a obra *Mãos à Obra: Aprendizado de Máquina com Scikit-Learn, Keras & TensorFlow*, de **Aurélien Géron**, que auxiliou na compreensão de conceitos relacionados a redes neurais e boas práticas de modelagem.

Essas ferramentas e materiais tiveram papel de suporte ao desenvolvimento, não substituindo a análise crítica e as decisões relacionadas à modelagem e aos experimentos.

---